# 00 — Pipeline uplift trọn gói (weak)

**Kịch bản weak:** YOLO đã train trên dữ liệu BDD của ta với **đủ class**, nhưng novel class **train thiếu**: base train chỉ có `weak_novel_images` ảnh chứa class đó (mặc định 100, smoke 10). Artifact riêng ở `art_weak/`.

Một notebook chạy cả luồng đánh giá uplift:

1. **Splits:** final test = BDD val. Dev, pool và base = BDD train, không giao nhau. Base train (dùng train YOLO) chứa đúng `weak_novel_images` ảnh có novel class, nhãn đầy đủ.
2. **Base model:** YOLO11n (COCO pretrained) train trên base train BDD với đủ class; novel class chỉ có ít ảnh nên yếu. Có thêm mốc uplift so với chính base model.
3. **Proposal:** mỗi thuật toán chọn K ảnh từ pool (BDD train, chưa từng dùng để train YOLO) mà không thấy nhãn, rồi chốt manifest.
   Sau đó lấy nhãn BDD của đúng các ảnh đó (simulated annotation). Bảng 1 so sánh tập dữ liệu do thuật toán tạo ra với tập random.
4. **Finetune:** không mở rộng head; finetune thẳng từ base model. Train trên replay + K ảnh mới. Mọi nhánh có cùng init, replay và số update.
5. **Đánh giá uplift** trên final test: AP novel, base mAP giữ được, hiệu từng cặp so với RANDOM cùng seed và cùng K.

**Gắn input:** dataset BDD100K. Có thể gắn thêm output của version trước để chạy tiếp, hoặc output NB1/NB2 để dùng lại splits và base model.
**GPU:** cần. **Internet:** bật.

**Hết giờ session / chạy tiếp:** Save Version → gắn output version vừa xong làm input của chính notebook này → chạy lại.
Mọi bước đã có kết quả được bỏ qua (manifest, base model, run, eval đều cache).

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
SCENARIO = "weak"   # missing: base chưa có novel class | weak: base có ít ảnh novel
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"   # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
# luôn lấy bản mới nhất của COMMIT từ GitHub (kể cả khi /tmp/repo còn từ lần chạy trước trong session tương tác)
subprocess.run(["git", "-C", R, "fetch", "-q", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", "-f", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
if f"{R}/src" not in sys.path:
    sys.path.insert(0, f"{R}/src")
for m in [m for m in sys.modules if m == "topicx" or m.startswith("topicx.")]:
    del sys.modules[m]   # chạy lại cell thì nạp lại code mới, không dùng module cũ trong kernel
import numpy as np, pandas as pd
from topicx import common as C, pipeline as P
cfg = C.load_config(SMOKE, SCENARIO)
commit = subprocess.run(["git", "-C", R, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("code:", commit)
print(f"kịch bản={SCENARIO} | artifact={cfg['art']} | copy từ input: {C.sync_inputs(cfg)} file")

In [ ]:
STAGE = "uplift"
# Test protocol (CPU, vài giây). Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE);

## Health check dataset
Vài giây, không đọc cả file nhãn: đường dẫn, số ảnh, định dạng nhãn, ảnh mẫu có vẽ box. Có mục FAIL thì notebook dừng ở đây.

In [ ]:
hc, sample = P.health_check(cfg)
if sample is not None:
    display(sample.resize((640, 360)))   # box đỏ phải ôm đúng vật thể

In [ ]:
NOVEL, REASON = "bus", "EDA ở NB1"   # đổi nếu chọn class khác (phải khớp splits trong input nếu dùng lại)
SEEDS = cfg["seeds"]
KS = cfg["ks"] if SMOKE else [250]          # số ảnh annotate mỗi nhánh
BUILTIN = ["RANDOM", "RETRIEVAL"]           # có sẵn: RANDOM, DIVERSITY, RETRIEVAL, RETRIEVAL_DIVERSE
REFERENCE = ["REPLAY_ONLY"]                 # thêm "ORACLE_POSITIVE", "RETRIEVAL_MATCHED" nếu cần (dùng nhãn ẩn)

## Cắm thuật toán proposal

**Cách A, viết hàm ngay tại đây:** `fn(pool, k, seed) -> list[id]` (hoặc `(list[id], meta_dict)`). Proposer chạy trong guard chặn đọc oracle và chỉ thấy:

| Thuộc tính / hàm | Nội dung |
|---|---|
| `pool.ids` | id ảnh pool (tên file BDD) |
| `pool.paths[id]` | đường dẫn ảnh |
| `pool.topic`, `pool.prompts` | tên class cần tìm và các câu mô tả |
| `pool.clip()` | CLIP embedding (N, 3 crop, D) |
| `pool.dino()` | DINOv2-S embedding (N, D) |
| `pool.topic_scores()` | điểm CLIP với topic |

**Cách B, dùng kết quả chạy ở nơi khác (ví dụ P-026):** dùng `splits/pool_ids.txt` và `splits/image_src.csv` trong output để biết pool gồm những ảnh nào. Xếp hạng ảnh pool, ghi ra `proposals/<tên>.txt` (mỗi dòng một id, tốt nhất xếp trước) hoặc `.csv` có cột `image`. Upload thành Kaggle Dataset rồi gắn vào. Nhánh sẽ tên là `EXT_<tên>`, lấy K id đầu tiên.

File ngoài phải được tạo **không dùng nhãn pool**. Pipeline chỉ kiểm được id có thuộc pool hay không.

In [ ]:
def my_proposer(pool, k, seed):
    """Ví dụ: retrieval nhưng bỏ ảnh quá giống nhau theo DINOv2 (thay bằng thuật toán của bạn)."""
    from topicx import select as S
    ids, relax = S.greedy_diverse(pool.ids, pool.order()[:5 * k], pool.dino(), k, min_dist=0.15)
    return ids, {"relax": relax}

MY = {}   # bật: {"MY_ALGO": my_proposer}
EXTERNAL = P.discover_ranked_files()
PROPOSERS = {b: P.PROPOSERS[b] for b in BUILTIN} | MY | EXTERNAL
BRANCHES = list(PROPOSERS) + REFERENCE
print("nhánh:", BRANCHES, "| seeds:", SEEDS, "| K:", KS)

## 1. Splits

In [ ]:
meta = P.prepare_splits(cfg, NOVEL, REASON)
display(pd.DataFrame(meta["report"]).T)
print("novel instances ở final test:", meta["test_novel_instances"])

## 2. Base model

In [ ]:
W, bm = P.train_base(cfg)
display(pd.DataFrame(bm["dev_ap"]).T)
print(f"ước tính train các nhánh: {P.estimate_hours(cfg, len(BRANCHES), SEEDS, KS):.1f} h GPU")

## 3. Proposal → chốt manifest → simulated annotation
Bảng: tập dữ liệu mỗi cách tạo ra (precision@K = tỉ lệ ảnh có novel).

In [ ]:
pool = P.Pool(cfg)
P.propose(cfg, PROPOSERS, SEEDS, KS, pool)
P.propose_privileged(cfg, REFERENCE, SEEDS, KS, pool)
ret = P.retrieval_eval(cfg, pool)
display(ret.groupby(["k", "branch"])[["precision_at_k", "recall_at_k", "novel_instances", "base_instances", "nn_cos_mean"]].mean())

## 4. Finetune từng nhánh

In [ ]:
todo = P.train_branches(cfg, BRANCHES, SEEDS, KS)
print("còn thiếu (chạy tiếp ở version sau):", todo or "không")

## 5. Đánh giá uplift trên final test

In [ ]:
res = P.evaluate(cfg, BRANCHES, SEEDS, KS)
print(f"base model: base mAP50-95 {res['base_map']:.4f}, AP novel {res['base_novel_ap']:.4f} | thiếu run: {res['missing'] or 'không'}")
print("Uplift AP50-95 novel so với RANDOM (từng seed):"); display(res["uplift"])
display(res["summary"])
print("báo cáo:", C.out(cfg, "eval/report.md"))

## Đọc kết quả

1. **Tập dữ liệu:** thuật toán có thu được nhiều ảnh và instance novel hơn RANDOM không? (`precision_at_k`, `novel_instances`)
2. **Uplift:** `uplift_novel_ap50_95` (so với RANDOM) có dương ở mọi seed không? `uplift_vs_base_model` = AP novel tăng so với base model. Đồng thời `base_map_delta` có xấu hơn `REPLAY_ONLY` không?
3. **Supervision tương đương:** cần nhánh `RETRIEVAL_MATCHED`. Nếu không chạy thì ghi là chưa kiểm.

Annotation là mô phỏng từ ground truth BDD. COCO pretrained đã có bus, motorcycle, person và bicycle, nên "novel" chỉ có nghĩa là mới với base model.